In [1]:
from pathlib import Path
import unicodedata
import csv
import math

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

from helpers import create_ids

In [2]:
path_data_es = "../../dwug_es_short/"
path_data_en = "../../dwug_en/data/"
target_words_es = [
    word.strip() for word in pd.read_csv(Path("../../test_data_es.csv"), sep="\t").word
]
file = open("../../target_words.txt", "r")
target_words_en = file.read().split("\n")
print(len(target_words_en))
file.close()

print(len(target_words_es))

TARGET_WORDS = {"dwug_es": target_words_es, "dwug_en": target_words_en}
PATH_TO_ANNOTATED_DATA = {"dwug_es": path_data_es, "dwug_en": path_data_en}


37
60


In [3]:
gold_data_spanish = pd.read_csv("../../test_data_es.csv", sep="\t")
gold_graded_change_data_spanish = gold_data_spanish.set_index("word")[
    "change_graded"
].to_dict()

with open(f"../../target_words.txt", "r") as f_in:
    target_words_english = f_in.read().split("\n")
    
gold_data_english = pd.read_csv("../../test_data_en.csv", sep="\t")
mask = gold_data_english["lemma"].isin(target_words_english)
gold_data_english = gold_data_english[mask]
gold_graded_change_data_english = gold_data_english.set_index("lemma")[
    "change_graded"
].to_dict()

GOLD_GRADED_CHANGE_DATA = {
    "dwug_es": gold_graded_change_data_spanish,
    "dwug_en": gold_graded_change_data_english,
}

gold_compare_score = gold_data_spanish.set_index("word")["COMPARE"].to_dict()
GOLD_COMPARE_SCORE = {
    "dwug_es": gold_compare_score
}
GOLD_COMPARE_SCORE["dwug_es"]

{'actitud': 2.24295774647887,
 'ataque': 2.57522123893805,
 'atrás': 2.9475,
 'ausencia': 3.38916256157635,
 'avance': 2.10688405797101,
 'banco': 1.26754385964912,
 'canal': 1.61346153846154,
 'capital': 2.15708812260536,
 'cobrar': 2.46969696969697,
 'colaborar': 2.59954751131222,
 'cólera': 1.625,
 'compasión': 3.73913043478261,
 'copiar': 2.44821428571429,
 'corriente': 1.76181102362205,
 'declinar': 2.21818181818182,
 'demá': 3.13598326359833,
 'diligencia': 2.25862068965517,
 'disco': 2.10810810810811,
 'distribuir': 3.24473684210526,
 'educado': 3.39230769230769,
 'elocuente': 3.39423076923077,
 'encargado': 3.3225,
 'enterar': 3.65885416666667,
 'especulación': 2.640625,
 'fallar': 2.60964912280702,
 'fallecimiento': 3.8510101010101,
 'historia': 3.19585253456221,
 'historiador': 3.9951690821256,
 'impulso': 2.95522388059701,
 'indicativo': 1.37740384615385,
 'juguete': 2.98936170212766,
 'maduro': 2.88297872340426,
 'maravilloso': 3.44675925925926,
 'marco': 1.19230769230769,


In [4]:
path_to_data = "../../input/{llm}/{dataset}/{prompt}"

In [5]:
for llm in ["llama3.1-8B", "mixtral-8xtb-v0.1"]:
    print(f"{llm}")

    for d in ["dwug_es", "dwug_en"]:
        print(f"  {d}:")

        for p in ["zs", "fs", "ct"]:
            print(f"    {p}:", end=" ")

            q = Path(path_to_data.format(llm=llm, dataset=d, prompt=p))
            assert q.exists() is True

            v1, v2 = [], []

            for tw in TARGET_WORDS[d]:
                tw_p = unicodedata.normalize("NFC", tw)

                data = pd.read_json(path_or_buf=[*q.glob(f"*{tw_p}.scores")][0])
                assert data.shape[0] != 0, f"problem loading data {tw_p}"
                
                mask = data["score"] == "-"
                data = data[~mask]

                if d == "dwug_en":
                    try:
                        data = create_ids(data)
                    except Exception:
                        print("error")
                        import sys

                        sys.exit(0)

                data["pair"] = data.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                try:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}data/{tw_p}/judgments.csv",
                        sep="\t",
                    )

                except Exception as e:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}{tw_p}/judgments.csv",
                        sep="\t",
                        engine="python",
                        quoting=csv.QUOTE_NONE,
                    )

                judgments = judgments[
                    ["identifier1", "identifier2", "judgment", "lemma"]
                ]
                assert judgments.shape[0] != 0, f"problem loading annotated for: {tw_p}"

                if d == "dwug_en":
                    judgments = create_ids(judgments)

                judgments["pair"] = judgments.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                judgments_median = (
                    judgments.groupby("pair")["judgment"].median().reset_index()
                )

                new_data = judgments_median.merge(data, on=["pair"])

                mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                    "identifier2"
                ].str.startswith("old")
                mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                    "identifier1"
                ].str.startswith("old")

                new_data = new_data[mask1 | mask2]

                assert (
                    len(new_data["judgment"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"
                assert (
                    len(new_data["score"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"

                v1.extend(new_data["judgment"].tolist())
                v2.extend(new_data["score"].tolist())
                # v2.extend(list(map(int, new_data["score"].tolist())))

                assert len(v1) == len(v2), f"problem vector's length: {tw_p}"

            spr, _ = spearmanr(v1, v2)

            print(spr)

llama3.1-8B
  dwug_es:
    zs: 0.012955364816830841
    fs: 0.06198607138986888
    ct: 0.015054662596029547
  dwug_en:
    zs: 0.08765107730900863
    fs: 0.10749085542176653
    ct: 0.04422524264604118
mixtral-8xtb-v0.1
  dwug_es:
    zs: 0.1545335995207829
    fs: 0.18410921868023233
    ct: 0.1556431483799669
  dwug_en:
    zs: 0.25925973548038156
    fs: 0.28332976802158844
    ct: 0.23481727069643643


In [12]:
for llm in ["llama3.1-8B", "mixtral-8xtb-v0.1"]:
    print(f"{llm}")

    for d in ["dwug_es", "dwug_en"]:
        print(f"  {d}:")

        for p in ["zs", "fs", "ct"]:
            print(f"    {p}:", end=" ")

            q = Path(path_to_data.format(llm=llm, dataset=d, prompt=p))
            assert q.exists() is True

            v1, v2 = [], []

            for tw in TARGET_WORDS[d]:
                tw_p = unicodedata.normalize("NFC", tw)

                data = pd.read_json(path_or_buf=[*q.glob(f"*{tw_p}.scores")][0])
                assert data.shape[0] != 0, f"problem loading data {tw_p}"

                mask = data["score"] == "-"
                data = data[~mask]

                if d == "dwug_en":
                    try:
                        data = create_ids(data)
                    except Exception:
                        print("error")
                        import sys

                        sys.exit(0)

                data["pair"] = data.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                try:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}data/{tw_p}/judgments.csv",
                        sep="\t",
                    )

                except Exception as e:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}{tw_p}/judgments.csv",
                        sep="\t",
                        engine="python",
                        quoting=csv.QUOTE_NONE,
                    )

                judgments = judgments[
                    ["identifier1", "identifier2", "judgment", "lemma"]
                ]
                assert judgments.shape[0] != 0, f"problem loading annotated for: {tw_p}"

                if d == "dwug_en":
                    judgments = create_ids(judgments)

                judgments["pair"] = judgments.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                judgments_median = (
                    judgments.groupby("pair")["judgment"].median().reset_index()
                )

                new_data = judgments_median.merge(data, on=["pair"])

                mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                    "identifier2"
                ].str.startswith("old")
                mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                    "identifier1"
                ].str.startswith("old")

                new_data = new_data[mask1 | mask2]

                assert (
                    len(new_data["judgment"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"
                assert (
                    len(new_data["score"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"

                # new_data["score_t"] = new_data.score.apply(int)

                v1.append(GOLD_GRADED_CHANGE_DATA[d][tw_p])
                v2.append(new_data["score"].mean())
                # v2.append(-1 * np.array(list(map(int, new_data["score"]))).mean())

                assert len(v1) == len(v2), f"problem vector's length: {tw_p}"

            spr, _ = spearmanr(v1, v2)

            print(spr)

llama3.1-8B
  dwug_es:
    zs: -0.09905936655286852
    fs: -0.26387183958537463
    ct: -0.0493729768540429
  dwug_en:
    zs: 0.8389203202150638
    fs: 0.8368447716942462
    ct: 0.7935616142298153
mixtral-8xtb-v0.1
  dwug_es:
    zs: 0.3904170606263353
    fs: -0.2312568865838792
    ct: -0.04251426688275254
  dwug_en:
    zs: 0.8453295825312389
    fs: 0.27967429232480734
    ct: -0.41024426100111927


In [7]:
for llm in ["llama3.1-8B", "mixtral-8xtb-v0.1"]:
    print(f"{llm}")

    for d in ["dwug_es"]:
        print(f"  {d}:")

        for p in ["zs", "fs", "ct"]:
            print(f"    {p}:", end=" ")

            q = Path(path_to_data.format(llm=llm, dataset=d, prompt=p))
            assert q.exists() is True

            v1, v2 = [], []

            for tw in TARGET_WORDS[d]:
                tw_p = unicodedata.normalize("NFC", tw)

                data = pd.read_json(path_or_buf=[*q.glob(f"*{tw_p}.scores")][0])
                assert data.shape[0] != 0, f"problem loading data {tw_p}"

                mask = data["score"] == "-"
                data = data[~mask]

                if d == "dwug_en":
                    try:
                        data = create_ids(data)
                    except Exception:
                        print("error")
                        import sys

                        sys.exit(0)

                data["pair"] = data.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                try:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}data/{tw_p}/judgments.csv",
                        sep="\t",
                    )

                except Exception as e:
                    judgments = pd.read_csv(
                        f"{PATH_TO_ANNOTATED_DATA[d]}{tw_p}/judgments.csv",
                        sep="\t",
                        engine="python",
                        quoting=csv.QUOTE_NONE,
                    )

                judgments = judgments[
                    ["identifier1", "identifier2", "judgment", "lemma"]
                ]
                assert judgments.shape[0] != 0, f"problem loading annotated for: {tw_p}"

                if d == "dwug_en":
                    judgments = create_ids(judgments)

                judgments["pair"] = judgments.apply(
                    lambda row: tuple(sorted([row["identifier1"], row["identifier2"]])),
                    axis=1,
                )

                judgments_median = (
                    judgments.groupby("pair")["judgment"].median().reset_index()
                )

                new_data = judgments_median.merge(data, on=["pair"])

                # mask1 = new_data["identifier1"].str.startswith("old") & ~new_data[
                #     "identifier2"
                # ].str.startswith("old")
                # mask2 = new_data["identifier2"].str.startswith("old") & ~new_data[
                #     "identifier1"
                # ].str.startswith("old")

                # new_data = new_data[mask1 | mask2]

                assert (
                    len(new_data["judgment"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"
                assert (
                    len(new_data["score"].tolist()) != 0
                ), f"vector's len is 0: {tw_p}"

                v1.append(GOLD_COMPARE_SCORE[d][tw_p])
                v2.append(np.array(list(map(int, new_data["score"].tolist()))).mean())

                assert len(v1) == len(v2), f"problem vector's length: {tw_p}"

            spr, _ = spearmanr(v1, v2)

            print(spr)

llama3.1-8B
  dwug_es:
    zs: 0.018560969173355188
    fs: 0.4344044791261003
    ct: 0.07852181161433733
mixtral-8xtb-v0.1
  dwug_es:
    zs: 0.12403445401500418
    fs: 0.16587941094748546
    ct: 0.10275076410113923


In [8]:
df = pd.DataFrame({"score": ['4', '3', '4', '3', '2', '1','4', '3']})

In [9]:
df["score"].mean()

5429017.875

In [10]:
a = [
    1.2031135907620939e265,
    2.5883890152588904e157,
    math.inf,
    math.inf,
    2.94334055916842e150,
    math.inf,
]

In [11]:
spearmanr([0.3,0.5,0.1,0.8, 0.5, 0.4], a)

SignificanceResult(statistic=-0.24641644145347896, pvalue=0.6378566719056556)